# 🏦 FTMO Crypto Portfolio Builder
**Goal**: Find the optimal multi-strategy, multi-asset crypto portfolio to pass the FTMO Challenge.

### FTMO Challenge Rules
| Rule | Value |
|------|-------|
| Account Size | $100,000 |
| Profit Target | 10% ($110,000) |
| Max Daily Loss | 5% ($5,000 from day start equity) |
| Max Total Loss | 10% ($90,000 absolute floor) |
| Time Limit | 30 calendar days |
| Min Trading Days | 4 |

### Pipeline
1. Download data for BTC, ETH, XRP, SOL, DOGE, AVAX
2. Run 3 strategies: MACD Crossover, Donchian Breakout, EMA Crossover
3. Grid search on in-sample, validate out-of-sample
4. FTMO Monte Carlo per strategy-asset pair
5. Portfolio combination: find best blend for max FTMO pass rate
6. Generate final tearsheet

## 0 · Install & Config

In [ ]:
%%capture
!pip install yfinance matplotlib seaborn pandas numpy scipy tqdm

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Edit these to control the run

TICKERS = ["BTC-USD", "ETH-USD", "XRP-USD", "SOL-USD", "DOGE-USD", "AVAX-USD"]
START_DATE   = "2018-01-01"
END_DATE     = None              # None = today
TRAIN_RATIO  = 0.60              # 60% IS / 40% OOS
INIT_CASH    = 100_000
FEES_PCT     = 0.0005            # 5 bps per side
SLIPPAGE_PCT = 0.0005            # 5 bps slippage
FREQ         = "1d"

# FTMO parameters
FTMO_ACCOUNT     = 100_000
FTMO_TARGET      = 110_000       # 10% profit target
FTMO_MAX_LOSS    = 90_000        # 10% max total loss floor
FTMO_DAILY_DD    = 0.05          # 5% daily drawdown limit
FTMO_DAYS        = 30            # 30 calendar days
FTMO_MC_SIMS     = 10_000        # Monte Carlo simulations

# Grid search granularity ("coarse" is fast, "fine" is thorough)
GRID_MODE = "coarse"  # "coarse" or "fine"

# Portfolio combo MC sims (can be expensive)
PORTFOLIO_MC_SIMS = 10_000

print(f"Config: {len(TICKERS)} tickers × 3 strategies")
print(f"Grid mode: {GRID_MODE}")
print(f"FTMO MC sims: {FTMO_MC_SIMS:,}")

## 1 · Data Download

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta
from itertools import product
from tqdm.auto import tqdm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

# Download all tickers
end = END_DATE or datetime.now().strftime("%Y-%m-%d")
data_dict = {}

for ticker in TICKERS:
    print(f"Downloading {ticker}...", end=" ")
    try:
        df = yf.download(ticker, start=START_DATE, end=end, interval=FREQ, progress=False)
        if len(df) < 500:
            print(f"SKIPPED (only {len(df)} bars)")
            continue
        # Flatten multi-index columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.columns = [c.lower() for c in df.columns]
        df = df[['open', 'high', 'low', 'close', 'volume']].dropna()
        df['returns'] = df['close'].pct_change()
        data_dict[ticker] = df
        print(f"OK — {len(df)} bars ({df.index[0].date()} to {df.index[-1].date()})")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\nLoaded {len(data_dict)} tickers successfully")

In [ ]:
# ── Train/Val Split ─────────────────────────────────────────────────
splits = {}
for ticker, df in data_dict.items():
    n = len(df)
    split_idx = int(n * TRAIN_RATIO)
    train = df.iloc[:split_idx].copy()
    val   = df.iloc[split_idx:].copy()
    splits[ticker] = {'train': train, 'val': val, 'full': df}
    print(f"{ticker}: IS {len(train)} bars ({train.index[0].date()} → {train.index[-1].date()}) | "
          f"OOS {len(val)} bars ({val.index[0].date()} → {val.index[-1].date()})")

## 2 · Strategy Definitions

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  STRATEGY SIGNAL GENERATORS
#  Each returns a Series of positions: 1 = long, 0 = flat
#  (Long-only for FTMO — no shorting on most prop firms for crypto)
# ══════════════════════════════════════════════════════════════════════

def macd_crossover_signal(df, fast_period, slow_period, signal_period):
    """MACD line crosses above/below signal line."""
    close = df['close']
    ema_fast = close.ewm(span=fast_period, adjust=False).mean()
    ema_slow = close.ewm(span=slow_period, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()

    pos = pd.Series(0, index=df.index)
    pos[macd_line > signal_line] = 1
    return pos


def donchian_breakout_signal(df, entry_period, exit_period, filter_period):
    """Enter on N-period high breakout, exit on M-period low.
    Optional trend filter: only trade if close > filter_period MA."""
    high_channel = df['high'].rolling(entry_period).max()
    low_channel  = df['low'].rolling(exit_period).min()
    trend_filter = df['close'].rolling(filter_period).mean()

    pos = pd.Series(0, index=df.index, dtype=float)
    in_trade = False

    for i in range(max(entry_period, filter_period), len(df)):
        if in_trade:
            if df['close'].iloc[i] < low_channel.iloc[i]:
                in_trade = False
                pos.iloc[i] = 0
            else:
                pos.iloc[i] = 1
        else:
            if (df['close'].iloc[i] > high_channel.iloc[i-1] and
                df['close'].iloc[i] > trend_filter.iloc[i]):
                in_trade = True
                pos.iloc[i] = 1
    return pos


def ema_crossover_signal(df, fast_period, slow_period, atr_filter_period=0):
    """EMA crossover with optional ATR-based volatility filter.
    If atr_filter_period > 0, only enter when ATR < 2x its own SMA (avoid chop)."""
    close = df['close']
    ema_fast = close.ewm(span=fast_period, adjust=False).mean()
    ema_slow = close.ewm(span=slow_period, adjust=False).mean()

    pos = pd.Series(0, index=df.index)
    pos[ema_fast > ema_slow] = 1

    if atr_filter_period > 0:
        tr = pd.concat([
            df['high'] - df['low'],
            (df['high'] - df['close'].shift(1)).abs(),
            (df['low']  - df['close'].shift(1)).abs()
        ], axis=1).max(axis=1)
        atr = tr.rolling(atr_filter_period).mean()
        atr_ma = atr.rolling(atr_filter_period * 2).mean()
        # Block entries in extreme volatility
        vol_block = atr > 2.0 * atr_ma
        pos[vol_block] = 0

    return pos


# ── Parameter grids ──────────────────────────────────────────────────
if GRID_MODE == "coarse":
    PARAM_GRIDS = {
        'MACD_Crossover': {
            'func': macd_crossover_signal,
            'params': {
                'fast_period':   [5, 8, 12, 15, 20, 25, 29],
                'slow_period':   [20, 26, 30, 35, 40, 45, 50, 55, 60],
                'signal_period': [3, 5, 7, 9, 12]
            }
        },
        'Donchian_Breakout': {
            'func': donchian_breakout_signal,
            'params': {
                'entry_period':  [7, 10, 14, 20, 30, 40, 50],
                'exit_period':   [3, 5, 8, 12, 17],
                'filter_period': [20, 30, 40, 55, 70, 90]
            }
        },
        'EMA_Crossover': {
            'func': ema_crossover_signal,
            'params': {
                'fast_period':       [5, 8, 12, 15, 20, 25, 30],
                'slow_period':       [20, 30, 40, 50, 60, 75, 100],
                'atr_filter_period': [0, 14, 20]
            }
        }
    }
else:  # fine
    PARAM_GRIDS = {
        'MACD_Crossover': {
            'func': macd_crossover_signal,
            'params': {
                'fast_period':   list(range(5, 31, 2)),
                'slow_period':   list(range(20, 61, 2)),
                'signal_period': list(range(3, 21))
            }
        },
        'Donchian_Breakout': {
            'func': donchian_breakout_signal,
            'params': {
                'entry_period':  list(range(7, 55, 3)),
                'exit_period':   list(range(3, 31, 2)),
                'filter_period': list(range(15, 101, 5))
            }
        },
        'EMA_Crossover': {
            'func': ema_crossover_signal,
            'params': {
                'fast_period':       list(range(3, 35, 2)),
                'slow_period':       list(range(15, 101, 3)),
                'atr_filter_period': [0, 10, 14, 20, 30]
            }
        }
    }

total_combos = sum(
    np.prod([len(v) for v in g['params'].values()])
    for g in PARAM_GRIDS.values()
)
print(f"Total grid combos per ticker: {total_combos:,}")
print(f"Total runs: {total_combos * len(data_dict):,}")

## 3 · Backtesting Engine

In [ ]:
def backtest(df, signal_func, params, init_cash=INIT_CASH,
             fees=FEES_PCT, slippage=SLIPPAGE_PCT):
    """
    Vectorised backtest: long-only, fully invested when signal=1.
    Returns dict of metrics + daily returns series.
    """
    pos = signal_func(df, **params)
    # Shift to avoid look-ahead: signal on close → trade next bar
    pos = pos.shift(1).fillna(0)

    rets = df['close'].pct_change().fillna(0)

    # Transaction costs on position changes
    trades = pos.diff().abs().fillna(0)
    cost = trades * (fees + slippage)

    strat_rets = pos * rets - cost
    equity = init_cash * (1 + strat_rets).cumprod()

    # Identify individual trades
    trade_entries = (pos.diff().fillna(0) == 1)
    trade_exits   = (pos.diff().fillna(0) == -1)

    # Count trades
    n_trades = trade_entries.sum()
    years = len(df) / 365.25

    # Compute trade-level returns
    trade_returns = []
    in_trade = False
    cum = 1.0
    for i in range(len(pos)):
        if pos.iloc[i] == 1:
            if not in_trade:
                cum = 1.0
                in_trade = True
            cum *= (1 + strat_rets.iloc[i])
        else:
            if in_trade:
                trade_returns.append(cum - 1)
                in_trade = False
    if in_trade:
        trade_returns.append(cum - 1)

    trade_returns = np.array(trade_returns) if trade_returns else np.array([0.0])
    wins = trade_returns[trade_returns > 0]
    losses = trade_returns[trade_returns <= 0]

    # Drawdown
    running_max = equity.expanding().max()
    dd = (equity - running_max) / running_max
    max_dd = dd.min()

    # Daily returns for FTMO sim
    daily_rets = strat_rets

    # Metrics
    ann_factor = 365  # crypto trades every day
    total_ret = equity.iloc[-1] / init_cash - 1
    ann_ret = (1 + total_ret) ** (ann_factor / len(df)) - 1 if len(df) > 0 else 0
    vol = strat_rets.std() * np.sqrt(ann_factor)
    sharpe = (strat_rets.mean() / strat_rets.std() * np.sqrt(ann_factor)) if strat_rets.std() > 0 else 0
    downside = strat_rets[strat_rets < 0].std()
    sortino = (strat_rets.mean() / downside * np.sqrt(ann_factor)) if downside > 0 else 0
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0

    wr = len(wins) / len(trade_returns) * 100 if len(trade_returns) > 0 else 0
    pf = wins.sum() / abs(losses.sum()) if len(losses) > 0 and losses.sum() != 0 else np.inf
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    payoff = abs(avg_win / avg_loss) if avg_loss != 0 else np.inf
    expectancy = np.mean(trade_returns) if len(trade_returns) > 0 else 0

    metrics = {
        'total_return': total_ret,
        'ann_return': ann_ret,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_dd': max_dd,
        'volatility': vol,
        'calmar': calmar,
        'trades': int(n_trades),
        'trades_yr': n_trades / years if years > 0 else 0,
        'win_rate': wr,
        'profit_factor': pf,
        'expectancy': expectancy,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'payoff': payoff,
    }

    return {
        'metrics': metrics,
        'daily_returns': daily_rets,
        'equity': equity,
        'positions': pos,
        'trade_returns': trade_returns
    }


# Quick sanity check
ticker0 = list(data_dict.keys())[0]
test_bt = backtest(data_dict[ticker0], macd_crossover_signal,
                   {'fast_period': 12, 'slow_period': 26, 'signal_period': 9})
print(f"Sanity check on {ticker0}: Sharpe={test_bt['metrics']['sharpe']:.3f}, "
      f"Trades={test_bt['metrics']['trades']}, Return={test_bt['metrics']['total_return']:.2%}")

## 4 · Grid Search (IS) + OOS Validation

In [ ]:
def grid_search(train_df, val_df, full_df, strategy_name, strategy_config, ticker):
    """
    Grid search on IS data, validate on OOS.
    Returns dict with best params + all metrics.
    """
    func = strategy_config['func']
    param_names = list(strategy_config['params'].keys())
    param_values = list(strategy_config['params'].values())
    all_combos = list(product(*param_values))

    best_sharpe = -np.inf
    best_params = None
    best_result = None
    results_log = []

    for combo in all_combos:
        params = dict(zip(param_names, combo))

        # Skip invalid combos
        if 'fast_period' in params and 'slow_period' in params:
            if params['fast_period'] >= params['slow_period']:
                continue

        try:
            res = backtest(train_df, func, params)
            m = res['metrics']

            # Filter: need minimum trades
            if m['trades'] < 5:
                continue

            results_log.append({**params, 'sharpe': m['sharpe'],
                                'return': m['total_return'], 'max_dd': m['max_dd'],
                                'trades': m['trades']})

            if m['sharpe'] > best_sharpe:
                best_sharpe = m['sharpe']
                best_params = params
                best_result = res
        except Exception:
            continue

    if best_params is None:
        return None

    # OOS validation
    oos_res  = backtest(val_df,  func, best_params)
    full_res = backtest(full_df, func, best_params)

    return {
        'strategy': strategy_name,
        'ticker': ticker,
        'best_params': best_params,
        'is_metrics': best_result['metrics'],
        'oos_metrics': oos_res['metrics'],
        'full_metrics': full_res['metrics'],
        'is_daily_returns': best_result['daily_returns'],
        'oos_daily_returns': oos_res['daily_returns'],
        'full_daily_returns': full_res['daily_returns'],
        'oos_trade_returns': oos_res['trade_returns'],
        'full_trade_returns': full_res['trade_returns'],
        'oos_equity': oos_res['equity'],
        'full_equity': full_res['equity'],
        'grid_log': pd.DataFrame(results_log),
        'combos_tested': len(all_combos),
    }

In [ ]:
# ── Run all grid searches ───────────────────────────────────────────
all_results = []
total_tasks = len(data_dict) * len(PARAM_GRIDS)

pbar = tqdm(total=total_tasks, desc="Grid Search")
for ticker, sp in splits.items():
    for strat_name, strat_cfg in PARAM_GRIDS.items():
        pbar.set_postfix_str(f"{strat_name} × {ticker}")
        result = grid_search(
            sp['train'], sp['val'], sp['full'],
            strat_name, strat_cfg, ticker
        )
        if result is not None:
            all_results.append(result)
        pbar.update(1)
pbar.close()

print(f"\n✅ Completed: {len(all_results)} strategy-asset pairs")

In [ ]:
# ── Results Summary Table ───────────────────────────────────────────
summary_rows = []
for r in all_results:
    ism = r['is_metrics']
    osm = r['oos_metrics']
    summary_rows.append({
        'Strategy': r['strategy'],
        'Ticker': r['ticker'],
        'IS Sharpe': round(ism['sharpe'], 3),
        'OOS Sharpe': round(osm['sharpe'], 3),
        'Sharpe Decay': round(1 - osm['sharpe']/ism['sharpe'], 2) if ism['sharpe'] > 0 else None,
        'IS Return': f"{ism['total_return']:.0%}",
        'OOS Return': f"{osm['total_return']:.0%}",
        'OOS MaxDD': f"{osm['max_dd']:.1%}",
        'OOS WinRate': f"{osm['win_rate']:.1f}%",
        'OOS PF': round(osm['profit_factor'], 2),
        'OOS Trades': osm['trades'],
        'OOS Trades/Yr': round(osm['trades_yr'], 1),
        'OOS Expect': f"{osm['expectancy']:.2%}",
        'Params': str(r['best_params']),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('OOS Sharpe', ascending=False)
summary_df.reset_index(drop=True, inplace=True)

print("\n" + "="*120)
print(" STRATEGY LEADERBOARD (sorted by OOS Sharpe)")
print("="*120)
display(summary_df)

## 5 · FTMO Monte Carlo Simulation (Per Strategy)

In [ ]:
def ftmo_monte_carlo(daily_returns, n_sims=FTMO_MC_SIMS, n_days=FTMO_DAYS,
                     account=FTMO_ACCOUNT, target=FTMO_TARGET,
                     max_loss=FTMO_MAX_LOSS, daily_dd_limit=FTMO_DAILY_DD):
    """
    Monte Carlo simulation of FTMO challenge.
    Resamples daily returns with replacement to simulate 30-day windows.
    """
    rets = daily_returns.dropna().values
    if len(rets) < 10:
        return {'pass_rate': 0, 'verdict': 'INSUFFICIENT_DATA'}

    passed = 0
    blown_total = 0
    blown_daily = 0
    still_trading = 0
    final_equities = []

    for _ in range(n_sims):
        # Sample n_days returns with replacement
        sampled = np.random.choice(rets, size=n_days, replace=True)
        equity = account
        peak = account
        alive = True
        hit_target = False

        for day_ret in sampled:
            day_start = equity
            equity *= (1 + day_ret)

            # Check daily DD (5% of day start equity)
            daily_loss = (day_start - equity) / day_start
            if daily_loss >= daily_dd_limit:
                blown_daily += 1
                alive = False
                break

            # Check total DD (equity < $90K)
            if equity < max_loss:
                blown_total += 1
                alive = False
                break

            # Check profit target
            if equity >= target:
                hit_target = True
                passed += 1
                break

        final_equities.append(equity)
        if alive and not hit_target:
            still_trading += 1

    pass_rate = passed / n_sims * 100
    final_equities = np.array(final_equities)

    if pass_rate >= 60:
        verdict = "STRONG"
    elif pass_rate >= 45:
        verdict = "POSSIBLE"
    elif pass_rate >= 30:
        verdict = "MARGINAL"
    else:
        verdict = "UNLIKELY"

    return {
        'pass_rate': round(pass_rate, 1),
        'verdict': verdict,
        'n_passed': passed,
        'n_blown_total': blown_total,
        'n_blown_daily': blown_daily,
        'n_still_trading': still_trading,
        'median_final_equity': round(np.median(final_equities), 2),
        'mean_final_equity': round(np.mean(final_equities), 2),
        'p25_equity': round(np.percentile(final_equities, 25), 2),
        'p75_equity': round(np.percentile(final_equities, 75), 2),
        'final_equities': final_equities,
    }

In [ ]:
# Run FTMO MC for each strategy-asset using OOS daily returns
ftmo_results = []

for r in tqdm(all_results, desc="FTMO Monte Carlo"):
    mc = ftmo_monte_carlo(r['oos_daily_returns'])
    r['ftmo'] = mc
    ftmo_results.append({
        'Strategy': r['strategy'],
        'Ticker': r['ticker'],
        'OOS Sharpe': round(r['oos_metrics']['sharpe'], 3),
        'FTMO Pass%': mc['pass_rate'],
        'Verdict': mc['verdict'],
        'Blown Daily%': round(mc['n_blown_daily']/FTMO_MC_SIMS*100, 1),
        'Blown Total%': round(mc['n_blown_total']/FTMO_MC_SIMS*100, 1),
        'Timed Out%': round(mc['n_still_trading']/FTMO_MC_SIMS*100, 1),
        'Median Equity': f"${mc['median_final_equity']:,.0f}",
    })

ftmo_df = pd.DataFrame(ftmo_results).sort_values('FTMO Pass%', ascending=False)
ftmo_df.reset_index(drop=True, inplace=True)

print("\n" + "="*110)
print(" FTMO CHALLENGE LEADERBOARD (sorted by pass rate)")
print("="*110)
display(ftmo_df)

## 6 · Portfolio Combination Engine

This is the key section. We combine the daily return streams from multiple strategy-asset pairs
using equal-weight or optimised allocations, then simulate the **combined** FTMO pass rate.

The idea: if Strategy A loses on Day X but Strategy B wins, the portfolio survives the daily DD limit.

In [ ]:
from itertools import combinations

def combine_daily_returns(results_subset, weights=None):
    """
    Combine daily returns from multiple strategy-asset pairs.
    Aligns on common dates, then takes weighted average.
    """
    n = len(results_subset)
    if weights is None:
        weights = np.ones(n) / n

    # Collect OOS daily return series
    all_rets = []
    for r in results_subset:
        s = r['oos_daily_returns'].copy()
        s.name = f"{r['strategy']}_{r['ticker']}"
        all_rets.append(s)

    # Align on common dates
    combined = pd.concat(all_rets, axis=1).fillna(0)

    # Weighted portfolio return
    portfolio_rets = combined.values @ weights
    return pd.Series(portfolio_rets, index=combined.index)


def score_combination(results_subset, weights=None, n_sims=PORTFOLIO_MC_SIMS):
    """Score a portfolio combination by its FTMO pass rate."""
    port_rets = combine_daily_returns(results_subset, weights)
    mc = ftmo_monte_carlo(port_rets, n_sims=n_sims)
    return mc

In [ ]:
# ── Filter viable candidates (OOS Sharpe > 0.3) ─────────────────────
viable = [r for r in all_results if r['oos_metrics']['sharpe'] > 0.3]
print(f"Viable candidates (OOS Sharpe > 0.3): {len(viable)}")
for v in viable:
    print(f"  {v['strategy']:20s} × {v['ticker']:10s} — "
          f"OOS Sharpe={v['oos_metrics']['sharpe']:.3f}, "
          f"FTMO={v['ftmo']['pass_rate']:.1f}%")

In [ ]:
# ── Test all 2, 3, and 4-asset combinations ─────────────────────────
combo_results = []

max_combo_size = min(5, len(viable))
total_combos = sum(
    len(list(combinations(range(len(viable)), k)))
    for k in range(2, max_combo_size + 1)
)
print(f"Testing {total_combos} portfolio combinations...")

pbar = tqdm(total=total_combos, desc="Portfolio Combos")
for k in range(2, max_combo_size + 1):
    for combo_idx in combinations(range(len(viable)), k):
        subset = [viable[i] for i in combo_idx]
        names = [f"{s['strategy'][:4]}_{s['ticker'][:3]}" for s in subset]

        # Equal weight
        mc = score_combination(subset, n_sims=PORTFOLIO_MC_SIMS)

        combo_results.append({
            'Portfolio': ' + '.join(names),
            'N': k,
            'Pass%': mc['pass_rate'],
            'Verdict': mc['verdict'],
            'Blown Daily%': round(mc['n_blown_daily']/PORTFOLIO_MC_SIMS*100, 1),
            'Blown Total%': round(mc['n_blown_total']/PORTFOLIO_MC_SIMS*100, 1),
            'Median $': mc['median_final_equity'],
            'components': combo_idx,
            'mc_result': mc,
        })
        pbar.update(1)
pbar.close()

combo_df = pd.DataFrame(combo_results).sort_values('Pass%', ascending=False)
combo_df_display = combo_df.drop(columns=['components', 'mc_result']).reset_index(drop=True)

print("\n" + "="*110)
print(" TOP 20 PORTFOLIO COMBINATIONS (by FTMO Pass Rate)")
print("="*110)
display(combo_df_display.head(20))

## 7 · Weight Optimization for Best Portfolio

In [ ]:
# Take the top combo and try optimizing weights
best_combo_row = combo_df.iloc[0]
best_combo_indices = best_combo_row['components']
best_subset = [viable[i] for i in best_combo_indices]
n_assets = len(best_subset)

print(f"Optimizing weights for: {best_combo_row['Portfolio']}")
print(f"Equal-weight pass rate: {best_combo_row['Pass%']:.1f}%")
print()

# Grid search over weight allocations (increments of 0.1)
def generate_weight_combos(n, step=0.1):
    """Generate all weight combinations that sum to 1.0."""
    levels = np.arange(step, 1.0 + step/2, step)
    combos = []
    for w in product(levels, repeat=n):
        if abs(sum(w) - 1.0) < 0.01:
            combos.append(np.array(w))
    return combos

weight_combos = generate_weight_combos(n_assets, step=0.1)
print(f"Testing {len(weight_combos)} weight combinations...")

best_pass = 0
best_weights = None
best_mc_opt = None
weight_log = []

for w in tqdm(weight_combos, desc="Weight Opt"):
    mc = score_combination(best_subset, weights=w, n_sims=5000)
    weight_log.append({'weights': w.tolist(), 'pass_rate': mc['pass_rate']})
    if mc['pass_rate'] > best_pass:
        best_pass = mc['pass_rate']
        best_weights = w
        best_mc_opt = mc

print(f"\n{'='*80}")
print(f" BEST PORTFOLIO ALLOCATION")
print(f"{'='*80}")
for i, s in enumerate(best_subset):
    print(f"  {s['strategy']:20s} × {s['ticker']:10s} → {best_weights[i]:.0%}")
print(f"\n  FTMO Pass Rate:      {best_pass:.1f}%")
print(f"  Verdict:             {best_mc_opt['verdict']}")
print(f"  Median Final Equity: ${best_mc_opt['median_final_equity']:,.0f}")
print(f"  Blown (Daily DD):    {best_mc_opt['n_blown_daily']/5000*100:.1f}%")
print(f"  Blown (Total DD):    {best_mc_opt['n_blown_total']/5000*100:.1f}%")

## 8 · Tearsheet & Visualization

In [ ]:
# ── Combined equity curve (OOS) ─────────────────────────────────────
port_rets = combine_daily_returns(best_subset, best_weights)
port_equity = FTMO_ACCOUNT * (1 + port_rets).cumprod()
port_dd = (port_equity - port_equity.expanding().max()) / port_equity.expanding().max()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

# Equity
ax1 = axes[0]
ax1.plot(port_equity.index, port_equity.values, color='#2196F3', linewidth=2, label='Portfolio (OOS)')
ax1.axhline(FTMO_ACCOUNT, color='gray', linestyle='--', alpha=0.5, label='Start $100K')

# Individual components
colors = ['#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#00BCD4']
for i, s in enumerate(best_subset):
    rets_i = s['oos_daily_returns'].reindex(port_rets.index).fillna(0)
    eq_i = FTMO_ACCOUNT * best_weights[i] * (1 + rets_i).cumprod()
    ax1.plot(eq_i.index, eq_i.values, color=colors[i % len(colors)],
             alpha=0.4, linewidth=1,
             label=f"{s['strategy'][:4]}_{s['ticker'][:3]} ({best_weights[i]:.0%})")

ax1.set_title(f"Portfolio Equity Curve (OOS) — Pass Rate: {best_pass:.1f}%",
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Equity ($)')
ax1.legend(loc='upper left', fontsize=9)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Drawdown
ax2 = axes[1]
ax2.fill_between(port_dd.index, port_dd.values * 100, 0, color='red', alpha=0.3)
ax2.plot(port_dd.index, port_dd.values * 100, color='red', linewidth=1)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.set_title('Portfolio Drawdown')

plt.tight_layout()
plt.show()

In [ ]:
# ── FTMO Monte Carlo Visualization ──────────────────────────────────
# Re-run with more sims for final plot
final_mc = score_combination(best_subset, best_weights, n_sims=FTMO_MC_SIMS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Equity paths
ax1 = axes[0]
rets_arr = port_rets.dropna().values
n_paths_to_show = 200

for _ in range(n_paths_to_show):
    sampled = np.random.choice(rets_arr, size=FTMO_DAYS, replace=True)
    equity_path = FTMO_ACCOUNT * np.cumprod(1 + sampled)
    equity_path = np.insert(equity_path, 0, FTMO_ACCOUNT)

    final = equity_path[-1]
    if final >= FTMO_TARGET:
        ax1.plot(equity_path, color='green', alpha=0.08, linewidth=0.8)
    elif min(equity_path) < FTMO_MAX_LOSS:
        ax1.plot(equity_path, color='red', alpha=0.08, linewidth=0.8)
    else:
        ax1.plot(equity_path, color='gray', alpha=0.08, linewidth=0.8)

ax1.axhline(FTMO_TARGET, color='green', linestyle='--', linewidth=2, label=f'Target ${FTMO_TARGET:,}')
ax1.axhline(FTMO_MAX_LOSS, color='red', linestyle='--', linewidth=2, label=f'Floor ${FTMO_MAX_LOSS:,}')
ax1.set_title(f'FTMO Challenge Paths — Pass Rate: {final_mc["pass_rate"]:.1f}%',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Trading Day')
ax1.set_ylabel('Equity ($)')
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Final equity distribution
ax2 = axes[1]
equities = final_mc['final_equities']
ax2.hist(equities, bins=80, color='#2196F3', alpha=0.7, edgecolor='white')
ax2.axvline(FTMO_TARGET, color='green', linestyle='--', linewidth=2, label='Target')
ax2.axvline(FTMO_MAX_LOSS, color='red', linestyle='--', linewidth=2, label='Floor')
ax2.axvline(np.median(equities), color='orange', linestyle='-', linewidth=2,
            label=f'Median: ${np.median(equities):,.0f}')
ax2.set_title('Final Equity Distribution', fontsize=13, fontweight='bold')
ax2.set_xlabel('Final Equity ($)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

# Print final stats
print(f"\n{'='*60}")
print(f" FINAL FTMO PORTFOLIO RESULTS")
print(f"{'='*60}")
print(f"  Pass Rate:           {final_mc['pass_rate']:.1f}%")
print(f"  Verdict:             {final_mc['verdict']}")
print(f"  Passed:              {final_mc['n_passed']:,}")
print(f"  Blown (Daily DD):    {final_mc['n_blown_daily']:,} ({final_mc['n_blown_daily']/FTMO_MC_SIMS*100:.1f}%)")
print(f"  Blown (Total DD):    {final_mc['n_blown_total']:,} ({final_mc['n_blown_total']/FTMO_MC_SIMS*100:.1f}%)")
print(f"  Timed Out:           {final_mc['n_still_trading']:,} ({final_mc['n_still_trading']/FTMO_MC_SIMS*100:.1f}%)")
print(f"  Median Equity:       ${final_mc['median_final_equity']:,.2f}")
print(f"  Mean Equity:         ${final_mc['mean_final_equity']:,.2f}")
print(f"  P25/P75 Equity:      ${final_mc['p25_equity']:,.2f} / ${final_mc['p75_equity']:,.2f}")

In [ ]:
# ── Correlation Heatmap ─────────────────────────────────────────────
all_oos_rets = []
labels = []
for r in viable:
    s = r['oos_daily_returns'].copy()
    s.name = f"{r['strategy'][:4]}_{r['ticker'][:3]}"
    all_oos_rets.append(s)
    labels.append(s.name)

corr_df = pd.concat(all_oos_rets, axis=1).corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, vmin=-0.3, vmax=0.8, square=True, ax=ax,
            linewidths=0.5)
ax.set_title('OOS Daily Return Correlations Between Strategy-Asset Pairs',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Lower correlation between portfolio components = better diversification for FTMO")

In [ ]:
# ── Component Detail Cards ──────────────────────────────────────────
print("\n" + "="*80)
print(" PORTFOLIO COMPONENT DETAILS")
print("="*80)

for i, s in enumerate(best_subset):
    m = s['oos_metrics']
    print(f"\n{'─'*60}")
    print(f" [{i+1}] {s['strategy']} × {s['ticker']}  —  Weight: {best_weights[i]:.0%}")
    print(f"{'─'*60}")
    print(f"  Parameters:    {s['best_params']}")
    print(f"  OOS Sharpe:    {m['sharpe']:.3f}")
    print(f"  OOS Return:    {m['total_return']:.2%}")
    print(f"  OOS MaxDD:     {m['max_dd']:.2%}")
    print(f"  OOS Win Rate:  {m['win_rate']:.1f}%")
    print(f"  OOS PF:        {m['profit_factor']:.2f}")
    print(f"  OOS Trades:    {m['trades']} ({m['trades_yr']:.1f}/yr)")
    print(f"  OOS Expect:    {m['expectancy']:.2%}")
    print(f"  Avg Win/Loss:  {m['avg_win']:.2%} / {m['avg_loss']:.2%}")
    print(f"  Payoff Ratio:  {m['payoff']:.2f}")
    print(f"  FTMO Pass%:    {s['ftmo']['pass_rate']:.1f}% ({s['ftmo']['verdict']})")

## 9 · Daily Return Characteristics (FTMO Risk Profile)

In [ ]:
# ── Analyse daily return characteristics for FTMO risk ──────────────
port_rets_clean = port_rets.dropna()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Daily returns histogram
ax = axes[0, 0]
ax.hist(port_rets_clean * 100, bins=80, color='#2196F3', alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.axvline(-5, color='red', linewidth=2, linestyle='--', label='FTMO 5% Daily Limit')
pct_beyond = (port_rets_clean < -0.05).mean() * 100
ax.set_title(f'Daily Return Distribution — {pct_beyond:.2f}% of days beyond 5% loss',
             fontsize=11)
ax.set_xlabel('Daily Return (%)')
ax.legend()

# Rolling 30-day return
ax = axes[0, 1]
roll_30 = (1 + port_rets_clean).rolling(30).apply(lambda x: x.prod() - 1, raw=True)
ax.plot(roll_30.index, roll_30.values * 100, color='#2196F3')
ax.axhline(10, color='green', linestyle='--', label='FTMO 10% Target')
ax.axhline(-10, color='red', linestyle='--', label='FTMO 10% Max Loss')
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_title('Rolling 30-Day Return (%)', fontsize=11)
ax.set_xlabel('Date')
ax.legend()

# Worst daily losses
ax = axes[1, 0]
worst_days = port_rets_clean.nsmallest(20) * 100
ax.barh(range(len(worst_days)), worst_days.values, color='red', alpha=0.7)
ax.set_yticks(range(len(worst_days)))
ax.set_yticklabels([d.strftime('%Y-%m-%d') for d in worst_days.index], fontsize=8)
ax.axvline(-5, color='red', linewidth=2, linestyle='--', label='5% Daily Limit')
ax.set_title('20 Worst Daily Losses (%)', fontsize=11)
ax.set_xlabel('Return (%)')
ax.legend()

# Monthly returns heatmap
ax = axes[1, 1]
monthly = port_rets_clean.resample('ME').apply(lambda x: (1+x).prod()-1) * 100
monthly_pivot = pd.DataFrame({
    'Year': monthly.index.year,
    'Month': monthly.index.month,
    'Return': monthly.values
}).pivot(index='Year', columns='Month', values='Return')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'][:len(monthly_pivot.columns)]
sns.heatmap(monthly_pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Return (%)'})
ax.set_title('Monthly Returns Heatmap (OOS Portfolio)', fontsize=11)

plt.tight_layout()
plt.show()

## 10 · Export Summary

In [ ]:
import json

# Build export JSON
export = {
    'metadata': {
        'run_date': datetime.now().isoformat(),
        'tickers': TICKERS,
        'strategies': list(PARAM_GRIDS.keys()),
        'train_ratio': TRAIN_RATIO,
        'grid_mode': GRID_MODE,
        'ftmo_mc_sims': FTMO_MC_SIMS,
    },
    'individual_results': [],
    'best_portfolio': {
        'components': [],
        'weights': best_weights.tolist(),
        'ftmo_pass_rate': final_mc['pass_rate'],
        'ftmo_verdict': final_mc['verdict'],
        'median_equity': final_mc['median_final_equity'],
        'blown_daily_pct': round(final_mc['n_blown_daily']/FTMO_MC_SIMS*100, 1),
        'blown_total_pct': round(final_mc['n_blown_total']/FTMO_MC_SIMS*100, 1),
    },
    'top10_combos': combo_df_display.head(10).to_dict('records'),
}

for i, s in enumerate(best_subset):
    export['best_portfolio']['components'].append({
        'strategy': s['strategy'],
        'ticker': s['ticker'],
        'weight': float(best_weights[i]),
        'params': s['best_params'],
        'oos_sharpe': round(s['oos_metrics']['sharpe'], 4),
        'oos_return': round(s['oos_metrics']['total_return'], 4),
        'oos_max_dd': round(s['oos_metrics']['max_dd'], 4),
        'ftmo_pass_rate': s['ftmo']['pass_rate'],
    })

for r in all_results:
    export['individual_results'].append({
        'strategy': r['strategy'],
        'ticker': r['ticker'],
        'params': r['best_params'],
        'is_sharpe': round(r['is_metrics']['sharpe'], 4),
        'oos_sharpe': round(r['oos_metrics']['sharpe'], 4),
        'oos_return': round(r['oos_metrics']['total_return'], 4),
        'oos_max_dd': round(r['oos_metrics']['max_dd'], 4),
        'oos_trades': r['oos_metrics']['trades'],
        'ftmo_pass_rate': r['ftmo']['pass_rate'],
        'ftmo_verdict': r['ftmo']['verdict'],
    })

# Save
with open('ftmo_portfolio_summary.json', 'w') as f:
    json.dump(export, f, indent=2, default=str)

print("✅ Summary saved to ftmo_portfolio_summary.json")
print(f"\nDownload: from google.colab import files; files.download('ftmo_portfolio_summary.json')")

In [ ]:
# ── Download helper ─────────────────────────────────────────────────
try:
    from google.colab import files
    files.download('ftmo_portfolio_summary.json')
except ImportError:
    print("Not in Colab — file saved locally.")

---
## Notes & Next Steps

### What this notebook does:
- Tests **MACD Crossover**, **Donchian Breakout**, and **EMA Crossover** across 6 crypto assets
- Grid searches parameters on in-sample data (60/40 split)
- Validates out-of-sample and runs FTMO Monte Carlo per strategy-asset pair
- Tests every 2/3/4/5-way portfolio combination for max FTMO pass rate
- Optimises allocation weights for the best combination

### Key FTMO insights from the model:
- **Daily DD is the killer** — most failures come from the 5% daily loss rule, not the 10% total
- **Diversification helps** — combining uncorrelated strategies smooths daily returns
- **Fewer, bigger winners** (high payoff ratio) work better than high win rate for FTMO

### To improve results:
1. Switch `GRID_MODE = "fine"` for more thorough parameter search
2. Add more tickers (LINK, MATIC, DOT, etc.)
3. Add more strategies (RSI mean-reversion, Bollinger breakout, Keltner channel)
4. Try position sizing: half-Kelly or volatility-target to reduce daily DD risk
5. Add intraday data (4H or 1H) for more trades per day
6. Walk-forward validation instead of single IS/OOS split